In [7]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
import torch as pt
from tqdm import tqdm
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from dataclasses import asdict
import os
import pandas as pd

from sklearn.model_selection import ParameterGrid

import scipy.stats.qmc as qm

from system_model import EAFModel, EAFParameters, step_eaf, TakeoutAnalysis

In [9]:
param_grid = {
    "O2_lance": [1.5, 2, 4, 6, 8],
    "P_arc": [30000, 35000, 40000, 55000, 75000],
    "O2_post": [0.5, 0.75, 1, 1.5, 2],
    "C_inj": [0.2, 0.25, 0.3, 0.5, 0.8],
    "FM_inj": [0.75, 1, 1.5, 2, 2.5],
    "DRI_add": [96, 110, 150],
}
sampler = qm.LatinHypercube(d=len(param_grid))

In [10]:
samples = sampler.random(n=10000) 
samples = qm.scale(
    samples,
    l_bounds=[min(v) for v in param_grid.values()],
    u_bounds=[max(v) for v in param_grid.values()],
)

In [11]:
samples_df = pd.DataFrame(
    samples, columns=list(param_grid.keys())
)
samples_df

,O2_lance,P_arc,O2_post,C_inj,FM_inj,DRI_add
0,7.195526,39712.365801,1.074769,0.568405,1.975600,147.216359
1,5.577911,39619.455045,1.908543,0.242822,2.412225,141.349774
2,4.162137,34433.518637,1.102698,0.352320,1.028932,126.364494
3,6.592072,39039.258099,1.703860,0.773217,2.148752,145.262738
4,5.609901,33706.390840,0.790499,0.687150,2.004735,108.081544
...,...,...,...,...,...,...
9995,7.223193,68702.749395,1.632363,0.561335,2.103960,102.834082
9996,1.507824,66580.394474,1.157394,0.575530,2.361154,139.834125
9997,2.960170,40956.251305,0.788574,0.433516,1.486460,116.158916
9998,4.683429,63356.842525,1.019342,0.212769,1.202911,116.832582


In [12]:
sampled_data = []
test_params = EAFParameters()
y_values = asdict(test_params)
y_values = [
    y_val
    for y_val in y_values.keys()
    if y_val.startswith("T_") or y_val.startswith("MX_")
]
with tqdm(total=samples_df.shape[0], iterable=samples_df.iterrows()) as gen:
    for sample_id, params in gen:
        print(f"Running sample {sample_id} with params: {params.to_dict()}")
        eaf_params = EAFParameters()
        eaf_params.secs = eaf_params.out + 1
        for k, v in params.items():
            setattr(eaf_params, k, v)
        eaf_params.x_names = params.keys()

        model = EAFModel(eaf_params)
        for s in tqdm(range(int(model.p.secs // model.p.ts))):
            step_eaf(model)

        sample = {
            **params.to_dict(),
            **{y: getattr(model.p, y) for y in y_values},
        }

        sampled_data.append(sample)
        gen.set_description(f"Sample ID: {sample_id}/{samples_df.shape[0]}")
        gen.refresh()
        gen.update(1)

  0%|          | 0/10000 [00:00<?, ?it/s]

Running sample 0 with params: {'O2_lance': 7.195525615796534, 'P_arc': 39712.36580106395, 'O2_post': 1.0747687681970177, 'C_inj': 0.5684046042134862, 'FM_inj': 1.9756004822864301, 'DRI_add': 147.21635896339882}


Sample ID: 0/10000:   0%|          | 1/10000 [00:26<73:25:03, 26.43s/it]

Running sample 1 with params: {'O2_lance': 5.577911230706062, 'P_arc': 39619.45504504258, 'O2_post': 1.9085433898349264, 'C_inj': 0.24282162818952496, 'FM_inj': 2.4122246654460495, 'DRI_add': 141.3497740839859}


Sample ID: 0/10000:   0%|          | 1/10000 [00:29<82:27:18, 29.69s/it]


KeyboardInterrupt: 

In [13]:
sample

{'O2_lance': 7.195525615796534,
 'P_arc': 39712.36580106395,
 'O2_post': 1.0747687681970177,
 'C_inj': 0.5684046042134862,
 'FM_inj': 1.9756004822864301,
 'DRI_add': 147.21635896339882,
 'T_DRI': 559.32,
 'MX_Fe_DRI': 0.88548,
 'MX_C_DRI': 0.016336,
 'MX_SiO2_DRI': 0.057,
 'MX_Al2O3_DRI': 0.03705,
 'MX_CaO_DRI': 0.00136,
 'MX_MgO_DRI': 0.0008,
 'MX_MnO_DRI': 0.0001,
 'MX_P2O5_DRI': 0.00186,
 'T_scr': 559.32,
 'MX_Fe_scr': 0.9705,
 'MX_C_scr': 0.004,
 'MX_Si_scr': 0.006,
 'MX_Cr_scr': 0.002,
 'MX_P_scr': 0.0005,
 'MX_Mn_scr': 0.006,
 'MX_comb_scr': 0.011,
 'T_slg': 300,
 'MX_CaO_slg': 0.573,
 'MX_MgO_slg': 0.415,
 'MX_SiO2_slg': 0.007,
 'MX_Al2O3_slg': 0.005,
 'MX_Mn_FM': 0.78,
 'MX_C_FM': 0.07,
 'MX_P_FM': 0.002,
 'MX_Si_FM': 0.003,
 'MX_Fe_FM': 0.145,
 'MX_SiO2_lSl': tensor(0.3930),
 'MX_Al2O3_lSl': tensor(0.2583),
 'MX_CaO_lSl': tensor(0.1070),
 'MX_MgO_lSl': tensor(0.0760),
 'MX_MnO_lSl': tensor(0.0958),
 'MX_P2O5_lSl': tensor(0.0138),
 'MX_Cr2O3_lSl': tensor(0.0002),
 'MX_FeO_lSl':

In [ ]:
sampled_data

[]

In [ ]:

sampled_df = pd.DataFrame(sampled_data)
sampled_df.to_csv("eaf_simulation_data.csv", index=False)